# Car Damage Severity — Fine-tune YOLO26 (Light / Moderate / Severe)

**Why not start from `car-damage.pt`:** that file is a YOLOv8-architecture checkpoint. YOLO26 changed the detection head significantly (removed Distribution Focal Loss, native NMS-free end-to-end inference) — its weights aren't compatible with YOLOv8's. So this notebook starts fine-tuning from Ultralytics' own **YOLO26 COCO-pretrained checkpoint** instead. You get the newer/faster architecture, at the cost of not carrying over your previous fine-tuning.

**Dataset approach is the same as before:** the Kaggle set is cropped classification images with no bounding boxes, so each image becomes a single full-frame detection box labeled `Light`/`Moderate`/`Severe`.

Model size used below: `yolo26s.pt` (small). Swap to `yolo26n.pt` (lighter/faster, good for edge/CPU) or `yolo26m.pt` / `yolo26l.pt` / `yolo26x.pt` (more capacity, slower) by changing `MODEL_NAME`.


## 1. Setup

In [ ]:
!pip install -q -U ultralytics kagglehub
import os, shutil, glob, yaml
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import kagglehub


## 2. Download the Kaggle dataset

In [ ]:
path = kagglehub.dataset_download("prajwalbhamere/car-damage-severity-dataset")
print("Path to dataset files:", path)

for root, dirs, files in os.walk(path):
    depth = root[len(path):].count(os.sep)
    if depth <= 3:
        indent = "  " * depth
        print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)" if files else f"{indent}{os.path.basename(root)}/")


## 3. Define classes and build the detection dataset

Class order here is arbitrary since we're starting from a fresh COCO-pretrained YOLO26 (not continuing your existing weights), so `Light=0, Moderate=1, Severe=2` is fine as-is.

In [ ]:
CLASS_TO_ID = {"Light": 0, "Moderate": 1, "Severe": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

CLASS_NAME_MAP = {
    "minor": "Light",
    "light": "Light",
    "moderate": "Moderate",
    "severe": "Severe",
}
SPLIT_MAP = {
    "training": "train", "train": "train",
    "testing": "val", "test": "val", "val": "val", "validation": "val",
}

def match_class(folder_name):
    name = folder_name.lower()
    for key, cls in CLASS_NAME_MAP.items():
        if key in name:
            return cls
    return None

def match_split(folder_name):
    name = folder_name.lower()
    for key, split in SPLIT_MAP.items():
        if key == name or key in name:
            return split
    return None

split_dirs = {}
for root, dirs, files in os.walk(path):
    for d in dirs:
        split = match_split(d)
        if split and split not in split_dirs:
            split_dirs[split] = os.path.join(root, d)

print("Detected split folders:", split_dirs)
assert "train" in split_dirs, "Couldn't auto-detect the training folder — check the tree above."


In [ ]:
DEST_ROOT = Path("/content/dataset_yolo26_det")
if DEST_ROOT.exists():
    shutil.rmtree(DEST_ROOT)

exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
BOX_FRACTION = 0.96  # full-frame synthetic box, small margin
counts = {}
skipped = 0

for split, split_path in split_dirs.items():
    img_dir = DEST_ROOT / "images" / split
    lbl_dir = DEST_ROOT / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for root, dirs, files in os.walk(split_path):
        cls_name = match_class(os.path.basename(root))
        if cls_name is None:
            continue
        class_id = CLASS_TO_ID[cls_name]

        for f in files:
            if not f.lower().endswith(exts):
                continue
            src = os.path.join(root, f)
            try:
                with Image.open(src) as im:
                    im.verify()
            except Exception:
                skipped += 1
                continue

            dst_img = img_dir / f
            if not dst_img.exists():
                shutil.copy2(src, dst_img)

            label_line = f"{class_id} 0.5 0.5 {BOX_FRACTION} {BOX_FRACTION}\n"
            dst_lbl = lbl_dir / (Path(f).stem + ".txt")
            dst_lbl.write_text(label_line)

            counts[(split, cls_name)] = counts.get((split, cls_name), 0) + 1

print("Labeled image counts:")
for k, v in sorted(counts.items()):
    print(k, v)
print("Skipped (unreadable) files:", skipped)


In [ ]:
# Carve a val split from train if none exists
if "val" not in split_dirs or not any((DEST_ROOT / "images" / "val").glob("*")):
    import random
    random.seed(42)
    train_img_dir = DEST_ROOT / "images" / "train"
    train_lbl_dir = DEST_ROOT / "labels" / "train"
    val_img_dir = DEST_ROOT / "images" / "val"
    val_lbl_dir = DEST_ROOT / "labels" / "val"
    val_img_dir.mkdir(parents=True, exist_ok=True)
    val_lbl_dir.mkdir(parents=True, exist_ok=True)

    by_class = {}
    for img_path in train_img_dir.glob("*"):
        lbl_path = train_lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue
        cls_id = lbl_path.read_text().split()[0]
        by_class.setdefault(cls_id, []).append(img_path)

    for cls_id, imgs in by_class.items():
        random.shuffle(imgs)
        n_val = max(1, int(0.2 * len(imgs)))
        for img_path in imgs[:n_val]:
            lbl_path = train_lbl_dir / (img_path.stem + ".txt")
            shutil.move(str(img_path), str(val_img_dir / img_path.name))
            shutil.move(str(lbl_path), str(val_lbl_dir / lbl_path.name))

    print("Carved out a validation split from train (20%).")


## 4. Write `data.yaml`

In [ ]:
names_list = [ID_TO_CLASS[i] for i in range(len(ID_TO_CLASS))]

data_yaml = {
    "path": str(DEST_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": names_list,
}

yaml_path = DEST_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(yaml_path.read_text())

for split in ["train", "val"]:
    n_img = len(list((DEST_ROOT / "images" / split).glob("*")))
    n_lbl = len(list((DEST_ROOT / "labels" / split).glob("*")))
    print(f"{split}: {n_img} images, {n_lbl} labels")


## 5. Fine-tune YOLO26

Downloads the pretrained checkpoint automatically on first use.

In [ ]:
MODEL_NAME = "yolo26s.pt"  # or yolo26n.pt / yolo26m.pt / yolo26l.pt / yolo26x.pt

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project="car_damage_yolo26",
    name="yolo26_run1",
    seed=42,
)


## 6. Evaluate

In [ ]:
metrics = model.val()
print(metrics)


## 7. Quick inference sanity check

In [ ]:
val_images = glob.glob(str(DEST_ROOT / "images" / "val" / "*"))[:6]
pred_results = model.predict(val_images, imgsz=640, conf=0.25)

for r in pred_results:
    if len(r.boxes) == 0:
        print(f"{os.path.basename(r.path)}: no detections")
        continue
    for box in r.boxes:
        cls_name = r.names[int(box.cls)]
        conf = float(box.conf)
        print(f"{os.path.basename(r.path)}: {cls_name} ({conf:.2%})")


## 8. Save fine-tuned weights

In [ ]:
best_weights = "car_damage_yolo26/yolo26_run1/weights/best.pt"
out_path = "/content/car-damage-yolo26-finetuned.pt"
shutil.copy2(best_weights, out_path)
print("Saved to:", out_path)

# from google.colab import files
# files.download(out_path)


## Notes

- Started from a fresh COCO-pretrained YOLO26 checkpoint, not your existing `car-damage.pt` — the architectures aren't weight-compatible, so this is a new fine-tune rather than a continuation.
- Same synthetic-full-frame-box caveat as the YOLOv8 version: boxes cover ~96% of each crop rather than a tight damage region, since the source data has no real bounding-box annotations.
- If accuracy from `yolo26s` isn't sufficient, try `yolo26m.pt` (more capacity) before adding more epochs — with this dataset size, epochs beyond diminishing returns risk overfitting more than under-capacity does.
